# Remaining distance to End-of-Life (RUL in km)

**remaining km = recent km/month × months until the SoH forecast reaches the EoL threshold.**

Degradation in this fleet is calendar/condition-driven, *not* mileage-driven (same km ≠ same SoH), so the battery ages out over **time** and kilometres accrue at the vehicle's usage rate — a high-utilisation vehicle delivers **more total km** before the same SoH threshold. EoL thresholds: 80% (end of first life / second-life trigger), 70%, 60% (true end of life). Model: `src/rul_km.py`; also surfaced live in the dashboard.

In [ ]:
import os, sys, warnings
from pathlib import Path
_r = Path.cwd()
while not (_r / 'requirements.txt').exists() and _r != _r.parent:
    _r = _r.parent
os.chdir(_r); sys.path.insert(0, str(_r / 'src')); sys.path.insert(0, str(_r / 'dashboard'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import rul_km as rk, build_dashboard as bd
print('loaded rul_km + build_dashboard')

## Per-vehicle remaining km — all three OEMs

Mahindra & Euler use the dashboard's condition-aware SoH forecast; Bajaj uses a √t reported-SoH extrapolation (no conditioned forecaster yet). km/month comes from the odometer.

In [ ]:
rows = []
for oem, builder in [('mahindra', bd.build_mahindra), ('euler', bd.build_euler)]:
    data = builder()
    feat = pd.read_parquet(f'data/{oem}/features/feature_table.parquet')
    kmpm = {v: rk.km_per_month(g['age_months'], g['odo_max']) for v, g in feat.groupby('vin')}
    for vin, v in data.items():
        rem = rk.remaining_km(v['obs'], v['fc'], kmpm.get(vin))
        rows.append(dict(oem=oem, vin=vin[-6:], soh=v['now'], kmpm=kmpm.get(vin) or 0,
                         rem80=rem[80], rem70=rem[70], rem60=rem[60]))
# Bajaj: sqrt-t reported-SoH forecast + km/month
b = pd.read_parquet('data/bajaj/features/feature_table.parquet').sort_values(['vin','age_months'])
for vin, g in b.groupby('vin'):
    g = g.reset_index(drop=True); a = g.age_months.to_numpy(); s = g.soh.to_numpy()
    A = np.c_[np.ones_like(a), np.sqrt(np.maximum(a,0))]; c, *_ = np.linalg.lstsq(A, s, rcond=None)
    if c[1] > 0: c = [s.mean(), 0.0]
    now_age = a[-1]; ms_now = int(g.month.iloc[-1].timestamp()*1000)
    fa = np.arange(now_age+1, now_age+121); fs = np.minimum.accumulate(np.clip(c[0]+c[1]*np.sqrt(fa), 0, 100))
    obs = [[ms_now, float(s[-1])]]
    fc = [[ms_now + int((age-now_age)*rk.MS_PER_MONTH), float(v)] for age, v in zip(fa, fs)]
    rem = rk.remaining_km(obs, fc, rk.km_per_month(g.age_months, g.odo_max))
    rows.append(dict(oem='bajaj', vin=vin[-6:], soh=float(s[-1]),
                     kmpm=rk.km_per_month(g.age_months, g.odo_max) or 0,
                     rem80=rem[80], rem70=rem[70], rem60=rem[60]))
R = pd.DataFrame(rows)
print('per-vehicle rows:', len(R))
def med_pos(x): x = x.dropna(); x = x[x>0]; return x.median() if len(x) else np.nan
summary = R.groupby('oem').agg(n=('vin','size'), km_month=('kmpm','median'),
    to80=('rem80', med_pos), to70=('rem70', med_pos), to60=('rem60', med_pos)).round(0)
summary

In [ ]:
# fleet median remaining-km by threshold (Mahindra mostly n/a — sparse odometer)
fig, ax = plt.subplots(figsize=(9, 4))
thr = ['rem80','rem70','rem60']; labels = ['to 80%','to 70%','to 60%']
x = np.arange(3); w = 0.26; cols = {'euler':'#1f9e8f','bajaj':'#e0922b','mahindra':'#5b6b7d'}
for i, oem in enumerate(['euler','bajaj','mahindra']):
    vals = [med_pos(R[R.oem==oem][t]) for t in thr]
    ax.bar(x + (i-1)*w, np.nan_to_num(vals), width=w, color=cols[oem], label=oem.capitalize())
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('median remaining km')
ax.set_title('Median remaining km to each EoL threshold'); ax.legend(frameon=False)
ax.spines[['top','right']].set_visible(False); plt.tight_layout()

In [ ]:
# per-vehicle detail (Euler — the cleanest coverage), sorted by remaining km to true EoL
R[R.oem=='euler'].sort_values('rem60', na_position='last')[['vin','soh','kmpm','rem80','rem70','rem60']].head(15)

### Caveats
- **Mahindra: n/a for most vehicles** — the odometer is logged too sparsely to derive a reliable km/month (the known late-logging issue), so remaining-km can't be computed for them.
- **Long extrapolations (to 60%) are low-confidence**, especially Bajaj (√t fit far past its ~10-month window).
- A `NaN`/blank remaining-km means the SoH forecast does not reach that threshold within its horizon (i.e. long remaining life), **or** km/month is unknown.
- This is `km/month × months-to-EoL`, deliberately **not** a claim that km drives degradation — it does not.